# WiFi CSI Captures — Data Cleaning & Labeling
**Before running**: Upload your `captures/` folder to Google Drive at `/MyDrive/fyp/captures/`

This notebook:
1. Loads all 29 capture CSVs
2. Parses raw CSI into amplitude + phase per subcarrier
3. Cleans problematic data (file #21, mixed csi_len)
4. Assigns people-count and localization labels from filenames
5. Saves two parquet files for fine-tuning:
   - `captures_people_counting.parquet` (256 subcarriers, interpolated)
   - `captures_localization.parquet` (128 subcarriers, with x/y/z labels)

In [ ]:
!pip install -q pyarrow

import numpy as np
import pandas as pd
import os, gc
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

CAPTURES_DIR = '/content/drive/MyDrive/fyp/captures'
SAVE_DIR = '/content/drive/MyDrive/fyp'
os.makedirs(SAVE_DIR, exist_ok=True)

assert os.path.isdir(CAPTURES_DIR), (
    f'Upload captures/ folder to {CAPTURES_DIR} first!'
)
files = [f for f in os.listdir(CAPTURES_DIR) if f.endswith('.csv')]
print(f'Found {len(files)} CSV files in {CAPTURES_DIR}')

In [ ]:
# ============================================================
# Complete metadata for all 29 capture files
# From WiFi_CSI_Dataset_Guide.docx
#
# Coordinate system (for localization):
#   Origin = ESP32 position
#   x = along LOS toward router (positive)
#   y = lateral (positive = right when facing router)
#   z = height above floor
# ============================================================

FILE_META = {
    # --- Group A: baseline, no person, no wall ---
    '1_meter_LOS_hotspot.csv':    {'num_people': 0, 'num_walls': 0},
    '2.1_meter_los_hotspot.csv':  {'num_people': 0, 'num_walls': 0},

    # --- Group B: 1 person on LOS, no wall (ESP-Router ~3m) ---
    '1_human_in_los_0.5m.csv':    {'num_people': 1, 'num_walls': 0, 'x': 0.5,  'y': 0.0,  'z': 1.0},
    '1_meter_Los_person.csv':     {'num_people': 1, 'num_walls': 0, 'x': 1.0,  'y': 0.0,  'z': 1.0},
    '1.5_meter_los_person.csv':   {'num_people': 1, 'num_walls': 0, 'x': 1.5,  'y': 0.0,  'z': 1.0},
    '2_meter_los_person.csv':     {'num_people': 1, 'num_walls': 0, 'x': 2.0,  'y': 0.0,  'z': 1.0},

    # --- Group C: 1 person off-axis, no wall ---
    '1m_off_LOS_left_0.5m.csv':   {'num_people': 1, 'num_walls': 0, 'x': 1.0,  'y': -0.5, 'z': 1.0},
    '1m_off_LOS_left_1m.csv':     {'num_people': 1, 'num_walls': 0, 'x': 1.0,  'y': -1.0, 'z': 1.0},
    '1m_off_LOS_right_0.5m.csv':  {'num_people': 1, 'num_walls': 0, 'x': 1.0,  'y': 0.5,  'z': 1.0},
    '1m_off_los_right_1m.csv':    {'num_people': 1, 'num_walls': 0, 'x': 1.0,  'y': 1.0,  'z': 1.0},
    '2m_off_los_left_0.5m.csv':   {'num_people': 1, 'num_walls': 0, 'x': 2.0,  'y': -0.5, 'z': 1.0},
    '2m_off_Los_left_1m.csv':     {'num_people': 1, 'num_walls': 0, 'x': 2.0,  'y': -1.0, 'z': 1.0},
    '2m_off_los_right_0.5m.csv':  {'num_people': 1, 'num_walls': 0, 'x': 2.0,  'y': 0.5,  'z': 1.0},
    '2m_off_Los_right_1m.csv':    {'num_people': 1, 'num_walls': 0, 'x': 2.0,  'y': 1.0,  'z': 1.0},

    # --- Group D: 2 walls baseline, no person ---
    '2_walls_LOS_2.5_m.csv': {'num_people': 0, 'num_walls': 2},
    '2_walls_4_m_LOS_last_100_needs_some_cleaning.csv': {
        'num_people': 0, 'num_walls': 2, 'clean_last_100': True
    },

    # --- Group E: 2 walls, 1 person stationary (2.5m path: router at x=2.5) ---
    '2_walls_2.5m_1_person_between_router_and_esp_in_los_midpoint.csv': {
        'num_people': 1, 'num_walls': 2, 'x': 1.25, 'y': 0.0, 'z': 1.0
    },
    '2_walls_2.5m_1_person_behind_the_router_by_1m_collinear_with_los.csv': {
        'num_people': 1, 'num_walls': 2, 'x': 3.5, 'y': 0.0, 'z': 1.0
    },
    '2_walls_2.5m_1_person_behind_router_by_2m_collinear_with_los.csv': {
        'num_people': 1, 'num_walls': 2, 'x': 4.5, 'y': 0.0, 'z': 1.0
    },
    '2_walls_2.5m_1_person_behind ESPby1m.csv': {
        'num_people': 1, 'num_walls': 2, 'x': -1.0, 'y': 0.0, 'z': 1.0
    },

    # --- Group E: 2 walls, 1 person stationary (4m path: router at x=4.0) ---
    '2_walls_4m_1_person_between_midpoint.csv': {
        'num_people': 1, 'num_walls': 2, 'x': 2.0, 'y': 0.0, 'z': 1.0
    },
    '2_walls_4m_1_person_behind_router_by_1m_collinear_with_los.csv': {
        'num_people': 1, 'num_walls': 2, 'x': 5.0, 'y': 0.0, 'z': 1.0
    },

    # --- Group F: 2 walls, dynamic / multi-person ---
    '2_walls_2.5m_1_person_walking_randomly_in_router_room.csv': {
        'num_people': 1, 'num_walls': 2
    },
    '2_walls_4m_2_persons_router_side_at_midpoint_esp_side_behind_by_1.2m.csv': {
        'num_people': 2, 'num_walls': 2
    },

    # --- Group G: 1 wall, sleeping scenario ---
    '1_wall_2_people_stationary_esp_3m_away_from_wall.csv': {
        'num_people': 2, 'num_walls': 1
    },
    '1_wall_2_person_station_esp_0.5m_away_from_router.csv': {
        'num_people': 2, 'num_walls': 1
    },
    '1_wall_2_person_stationary_1_person_in_LOS_esp_3m_away_from_wall.csv': {
        'num_people': 3, 'num_walls': 1
    },

    # --- Group H: no wall, walking ---
    'walking_in_Square_shape.csv': {'num_people': 1, 'num_walls': 0},
    'walking_randomly_through_the_whole_room.csv': {'num_people': 1, 'num_walls': 0},
}

print(f'Defined metadata for {len(FILE_META)} files')
# Verify all files exist
for fname in FILE_META:
    fpath = os.path.join(CAPTURES_DIR, fname)
    if not os.path.exists(fpath):
        print(f'  WARNING: {fname} not found!')

In [ ]:
# ============================================================
# Load all CSVs, parse csi_raw -> amplitude + phase
# Keep only csi_len=256 rows (128 subcarriers)
# ============================================================

all_amp = []     # will be (N, 128)
all_phase = []   # will be (N, 128)
all_labels = []  # list of dicts
skipped_len = 0
skipped_parse = 0

for fname, meta in FILE_META.items():
    fpath = os.path.join(CAPTURES_DIR, fname)
    if not os.path.exists(fpath):
        continue

    df = pd.read_csv(fpath)

    # Clean file #21: remove last 100 rows
    if meta.get('clean_last_100'):
        before = len(df)
        df = df.iloc[:-100]
        print(f'  Cleaned {fname}: {before} -> {len(df)} rows')

    # Filter to csi_len=256 only (128 subcarriers)
    mask = df['csi_len'] == 256
    skipped_len += (~mask).sum()
    df = df[mask].reset_index(drop=True)

    # Parse each row
    for idx in range(len(df)):
        row = df.iloc[idx]
        try:
            vals = np.array(str(row['csi_raw']).split(), dtype=np.float32)
            if len(vals) < 256:
                skipped_parse += 1
                continue
            vals = vals[:256]
            I = vals[0::2]   # 128 in-phase values
            Q = vals[1::2]   # 128 quadrature values
            amp = np.sqrt(I**2 + Q**2)
            phase = np.arctan2(Q, I)
            all_amp.append(amp)
            all_phase.append(phase)
            all_labels.append({
                'filename': fname,
                'rssi': int(row['rssi']),
                'mac': str(row['mac']),
                'num_people': meta['num_people'],
                'num_walls': meta['num_walls'],
                'x': meta.get('x', np.nan),
                'y': meta.get('y', np.nan),
                'z': meta.get('z', np.nan),
            })
        except Exception as e:
            skipped_parse += 1

X_amp = np.stack(all_amp).astype(np.float32)    # (N, 128)
X_phase = np.stack(all_phase).astype(np.float32) # (N, 128)
meta_df = pd.DataFrame(all_labels)
del all_amp, all_phase, all_labels; gc.collect()

print(f'\nTotal rows: {len(meta_df):,}')
print(f'Skipped (csi_len!=256): {skipped_len}')
print(f'Skipped (parse errors): {skipped_parse}')
print(f'Amplitude shape: {X_amp.shape}')
print(f'Phase shape: {X_phase.shape}')

In [ ]:
# ============================================================
# Data overview & sanity checks
# ============================================================

print('=== People Count Distribution ===')
pc_dist = meta_df['num_people'].value_counts().sort_index()
for k, v in pc_dist.items():
    print(f'  {k} people: {v:,} rows ({100*v/len(meta_df):.1f}%)')

print(f'\n=== Wall Distribution ===')
print(meta_df['num_walls'].value_counts().sort_index().to_string())

print(f'\n=== Files per AP (MAC) ===')
print(meta_df.groupby('mac')['filename'].nunique().to_string())

# Localization subset
has_pos = meta_df['x'].notna()
print(f'\n=== Localization subset: {has_pos.sum():,} rows from {meta_df[has_pos]["filename"].nunique()} files ===')
print(f'  x range: {meta_df.loc[has_pos, "x"].min():.1f} to {meta_df.loc[has_pos, "x"].max():.1f} m')
print(f'  y range: {meta_df.loc[has_pos, "y"].min():.1f} to {meta_df.loc[has_pos, "y"].max():.1f} m')

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# People count
pc_dist.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('People Count Distribution')
axes[0].set_xlabel('People'); axes[0].set_ylabel('Rows')

# Sample amplitude spectra by people count
for n in sorted(meta_df['num_people'].unique()):
    idx = meta_df[meta_df['num_people'] == n].index[0]
    axes[1].plot(X_amp[idx], label=f'{n} people', alpha=0.7)
axes[1].set_title('Sample Amplitude Spectra')
axes[1].set_xlabel('Subcarrier'); axes[1].legend()

# Localization positions (x vs y)
loc_meta = meta_df[has_pos]
axes[2].scatter(loc_meta['x'], loc_meta['y'], s=1, alpha=0.3)
# Mark unique positions
unique_pos = loc_meta.groupby('filename')[['x', 'y']].first()
axes[2].scatter(unique_pos['x'], unique_pos['y'], s=100, c='red', marker='x', zorder=5)
axes[2].set_title(f'Localization Positions ({len(unique_pos)} unique)')
axes[2].set_xlabel('x (m)'); axes[2].set_ylabel('y (m)')
axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Create PEOPLE COUNTING dataset
# Interpolate 128 -> 256 subcarriers to match pre-trained model
# ============================================================

x_128 = np.arange(128)
x_256 = np.linspace(0, 127, 256)

print('Interpolating 128 -> 256 subcarriers...')
X_amp_256 = np.array(
    [np.interp(x_256, x_128, row) for row in X_amp],
    dtype=np.float32
)
X_phase_256 = np.array(
    [np.interp(x_256, x_128, row) for row in X_phase],
    dtype=np.float32
)
print(f'Interpolated: amp {X_amp_256.shape}, phase {X_phase_256.shape}')

# Build dataframe
pc_data = {}
for i in range(256):
    pc_data[f'amp_{i}'] = X_amp_256[:, i]
    pc_data[f'phase_{i}'] = X_phase_256[:, i]
pc_data['num_people'] = meta_df['num_people'].values
pc_data['num_walls'] = meta_df['num_walls'].values
pc_data['filename'] = meta_df['filename'].values

pc_df = pd.DataFrame(pc_data)
pc_path = f'{SAVE_DIR}/captures_people_counting.parquet'
pc_df.to_parquet(pc_path, index=False)
print(f'\nSaved: {pc_path}')
print(f'  Shape: {pc_df.shape}')
print(f'  Size: {os.path.getsize(pc_path)/1e6:.1f} MB')
print(f'  Classes: {dict(pc_df["num_people"].value_counts().sort_index())}')

# Verify interpolation visually
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
idx = 0
axes[0].plot(x_128, X_amp[idx], 'b-', label='Original 128')
axes[0].plot(x_256 * 128/256, X_amp_256[idx], 'r--', alpha=0.7, label='Interpolated 256')
axes[0].set_title('Amplitude: Original vs Interpolated')
axes[0].legend(); axes[0].set_xlabel('Subcarrier')

axes[1].plot(x_128, X_phase[idx], 'b-', label='Original 128')
axes[1].plot(x_256 * 128/256, X_phase_256[idx], 'r--', alpha=0.7, label='Interpolated 256')
axes[1].set_title('Phase: Original vs Interpolated')
axes[1].legend(); axes[1].set_xlabel('Subcarrier')
plt.tight_layout(); plt.show()

del X_amp_256, X_phase_256, pc_data, pc_df; gc.collect()
print('Done!')

In [ ]:
# ============================================================
# Create LOCALIZATION dataset
# 128 subcarriers (native), only rows with known x/y/z
# ============================================================

has_pos = meta_df['x'].notna()
loc_indices = np.where(has_pos.values)[0]
loc_amp = X_amp[loc_indices]      # (N_loc, 128)
loc_phase = X_phase[loc_indices]   # (N_loc, 128)
loc_meta = meta_df.iloc[loc_indices].reset_index(drop=True)

# Build dataframe
loc_data = {}
for i in range(128):
    loc_data[f'amp_{i}'] = loc_amp[:, i]
    loc_data[f'phase_{i}'] = loc_phase[:, i]
loc_data['x'] = loc_meta['x'].values.astype(np.float32)
loc_data['y'] = loc_meta['y'].values.astype(np.float32)
loc_data['z'] = loc_meta['z'].values.astype(np.float32)
loc_data['num_walls'] = loc_meta['num_walls'].values
loc_data['filename'] = loc_meta['filename'].values

loc_df = pd.DataFrame(loc_data)
loc_path = f'{SAVE_DIR}/captures_localization.parquet'
loc_df.to_parquet(loc_path, index=False)
print(f'Saved: {loc_path}')
print(f'  Shape: {loc_df.shape}')
print(f'  Size: {os.path.getsize(loc_path)/1e6:.1f} MB')
print(f'  Unique positions: {loc_df.groupby(["x", "y", "z"]).ngroups}')
print(f'  Files: {loc_df["filename"].nunique()}')

# Show position breakdown
print(f'\n=== Rows per position ===')
pos_counts = loc_df.groupby('filename')[['x', 'y', 'num_walls']].agg(
    x=('x', 'first'), y=('y', 'first'),
    walls=('num_walls', 'first'), rows=('num_walls', 'count')
)
# Simpler approach
for fname in sorted(loc_df['filename'].unique()):
    sub = loc_df[loc_df['filename'] == fname]
    print(f'  {fname[:50]:50s} | x={sub["x"].iloc[0]:5.1f} y={sub["y"].iloc[0]:5.1f} | walls={sub["num_walls"].iloc[0]} | {len(sub):,} rows')

In [ ]:
# ============================================================
# Summary
# ============================================================

print('=' * 60)
print('CLEANING COMPLETE')
print('=' * 60)
print(f'\nSaved to {SAVE_DIR}/:')
for fname in ['captures_people_counting.parquet', 'captures_localization.parquet']:
    fpath = os.path.join(SAVE_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        print(f'  {fname} ({size/1e6:.1f} MB)')

print(f'\nPeople Counting dataset:')
print(f'  {len(meta_df):,} rows x 512 features (256 amp + 256 phase)')
print(f'  Classes: 0, 1, 2, 3 people')

print(f'\nLocalization dataset:')
print(f'  {has_pos.sum():,} rows x 256 features (128 amp + 128 phase)')
print(f'  {meta_df[has_pos]["filename"].nunique()} positions (18 files)')
print(f'  x: [{meta_df.loc[has_pos, "x"].min():.1f}, {meta_df.loc[has_pos, "x"].max():.1f}] m')
print(f'  y: [{meta_df.loc[has_pos, "y"].min():.1f}, {meta_df.loc[has_pos, "y"].max():.1f}] m')
print(f'\nNext: Run captures_finetune.ipynb to fine-tune the models')